# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their `@id`.

### Dataset Source
The dataset source is provided as a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# The metadata object for this dataset
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n\nPublished: {metadata.datePublished}")

## 2. Data Overview
List all available record sets, their field IDs (using `@id`), and information about the fields/columns.

We'll display:
- The available RecordSets (`@id`s and names)
- For each RecordSet, its Fields (`@id`, name, and type)


In [ ]:
# Explore available record sets and their fields
print("Record Sets available in metadata (referenced by their @id):\n")
record_set_ids = []
for rs in metadata.recordSets:
    print(f"@id: {rs.id}\n  Name: {rs.name}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id.ljust(35)} Name: {field.name.ljust(20)} Type: {field.dataType if hasattr(field, 'dataType') else 'unknown'}")
    print("---")
print("\nRecord Set IDs found:", record_set_ids)

## 3. Data Extraction
Load all available record sets into pandas DataFrames using their `@id`.

- Use the record set and field `@id`s from the data overview for extraction.
- All data extraction and reference use variable names for repeatable analysis.


In [ ]:
# Extract all record sets as DataFrames using @id
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set @id: {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Shape: {df.shape}")
    print(df.head(2))
    print("-"*40)
assert len(dataframes) > 0, "No record sets found."
# For demonstration, pick the first available record set for further analysis
main_rs_id = record_set_ids[0]
print(f"\nMain record set for EDA: {main_rs_id}\n")
main_df = dataframes[main_rs_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping. Choose fields for the demo based on the columns present in the main record set. All fields must be referenced by their `@id`.


In [ ]:
# Show sample of columns (@id) and pick fields for analysis
main_columns = main_df.columns.tolist()
print("Columns for main record set (referenced by @id):", main_columns)

# Guess a likely numeric field by common proteomics conventions
possible_numeric_fields = [col for col in main_columns if any(x in col.lower() for x in ['abundance', 'count', 'coverage', 'mw', 'weight', 'score'])]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = main_columns[1]  # arbitrary fallback
print(f"Using numeric field for filtering and normalization: {numeric_field_id}")

# Filter records based on a threshold
threshold = main_df[numeric_field_id].quantile(0.75) if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else None
if threshold is not None:
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (upper quartile filter): {len(filtered_df)} records")
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Cannot filter: field {numeric_field_id} is not numeric.")

# Attempt to group by a categorical field if available (e.g., condition, sample, or description)
possible_group_fields = [col for col in main_columns if any(x in col.lower() for x in ['condition', 'sample', 'group', 'type', 'category', 'description'])]
if possible_group_fields and threshold is not None:
    group_field_id = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped result of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field found or no numeric field for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, group-wise means.
All axes/labels show field `@id`s for clarity.

In [ ]:
import matplotlib.pyplot as plt

if threshold is not None:
    # Histogram of the numeric field
    main_df[numeric_field_id].hist(bins=30, edgecolor='k')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot
    main_df[[numeric_field_id]].boxplot()
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()

    # If we grouped, show as bar chart
    if 'grouped_df' in locals():
        grouped_df.plot.bar(x=group_field_id, y=numeric_field_id, legend=False)
        plt.ylabel(numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()


## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, examine, and analyze the FAIR^2 dataset for human mast cell-derived extracellular vesicle proteomics. We referenced all record sets and fields by their `@id`, loaded them into pandas DataFrames, performed basic filtering and normalization on a numeric field, and visualized the results. For deeper analysis, refer to the dataset's Croissant metadata for detailed field semantics and data provenance.